# Chapter 3 Companion Notebook: Financial Markets, Institutions, and Instruments

This notebook reproduces the quantitative examples from Chapter 3 of *AI in Finance* (`chapter3.tex`): the bid-ask spread as a measure of trading cost, walking the limit order book to see price impact from order size, and the simple no-arbitrage argument connecting a risk-free bond to the price of a certain-payoff claim.

## 1. The bid-ask spread

The bid-ask spread is the simplest measure of the cost of trading immediately, rather than waiting for a better price.

In [1]:
bid = 100.20
ask = 100.35

spread = ask - bid
mid = (bid + ask) / 2
spread_bps = spread / mid * 10_000

print(f"Bid: ${bid:.2f}, Ask: ${ask:.2f}")
print(f"Spread: ${spread:.2f} ({spread_bps:.1f} bps of the mid price)")

Bid: $100.20, Ask: $100.35
Spread: $0.15 (15.0 bps of the mid price)


A round-trip trade (buy then immediately sell) at the quoted prices costs the full spread. Compare a liquid, tightly-quoted stock to an illiquid one:

In [2]:
import pandas as pd

quotes = pd.DataFrame({
    'Bid': [100.20, 24.90, 8.10],
    'Ask': [100.35, 25.30, 8.70],
}, index=['Liquid large-cap', 'Mid-cap', 'Illiquid small-cap'])

quotes['Spread'] = quotes['Ask'] - quotes['Bid']
quotes['Mid'] = (quotes['Bid'] + quotes['Ask']) / 2
quotes['Spread (bps)'] = quotes['Spread'] / quotes['Mid'] * 10_000
quotes.round(2)

,Bid,Ask,Spread,Mid,Spread (bps)
Liquid large-cap,100.2,100.35,0.15,100.28,14.96
Mid-cap,24.9,25.30,0.40,25.10,159.36
Illiquid small-cap,8.1,8.70,0.60,8.40,714.29


Notice how much larger the relative (basis-point) spread is for the illiquid small-cap name, even though its dollar spread looks modest -- this is exactly why comparing raw dollar spreads across differently priced stocks can be misleading, and why practitioners usually compare spreads in basis points.

## 2. Walking the limit order book

A large market order can consume several price levels, receiving a progressively worse average price than the best quote alone would suggest.

In [3]:
order_book_asks = pd.DataFrame({
    "Price": [50.10, 50.15, 50.20],
    "Size": [100, 200, 300],
}, index=[1, 2, 3])

best_bid, best_ask = 50.05, 50.10
quoted_spread = best_ask - best_bid
print(f"Best bid: ${best_bid:.2f}, Best ask: ${best_ask:.2f}, Quoted spread: ${quoted_spread:.2f}")

# A market buy order for 250 shares walks through two ask levels
order_size = 250
filled = 0
cost = 0.0
for price, size in zip(order_book_asks["Price"], order_book_asks["Size"]):
    take = min(size, order_size - filled)
    cost += take * price
    filled += take
    if filled >= order_size:
        break

avg_execution_price = cost / order_size
price_impact_bps = (avg_execution_price - best_ask) / best_ask * 10_000
print(f"Average execution price for a {order_size}-share buy: ${avg_execution_price:.2f}")
print(f"Price impact above best ask: {price_impact_bps:.1f} bps")

Best bid: $50.05, Best ask: $50.10, Quoted spread: $0.05
Average execution price for a 250-share buy: $50.13
Price impact above best ask: 6.0 bps


## 3. A simple no-arbitrage example

A one-period risk-free bond costs \$100 today and pays \$105 in one year.

In [4]:
bond_price = 100
bond_payoff = 105
risk_free_rate = (bond_payoff - bond_price) / bond_price
print(f"Implied risk-free rate: {risk_free_rate:.2%}")

Implied risk-free rate: 5.00%


Now suppose a separate claim pays exactly \$1 with certainty in one year (a zero-coupon bond with face value \$1). No-arbitrage says its price must be consistent with the same risk-free rate.

In [5]:
no_arbitrage_price = 1 / (1 + risk_free_rate)
print(f"No-arbitrage price of the $1 claim: ${no_arbitrage_price:.4f}")

# Suppose the claim is mispriced in the market at $0.95
market_price = 0.95
implied_return = (1 - market_price) / market_price
print(f"Market price: ${market_price:.2f}")
print(f"Implied return if bought at the market price: {implied_return:.2%}")
print(f"Risk-free rate available from the bond:        {risk_free_rate:.2%}")

if implied_return > risk_free_rate:
    print("Arbitrage: buy the claim, fund it by borrowing at the risk-free rate, "
          "and pocket the difference risk-free.")

No-arbitrage price of the $1 claim: $0.9524
Market price: $0.95
Implied return if bought at the market price: 5.26%
Risk-free rate available from the bond:        5.00%
Arbitrage: buy the claim, fund it by borrowing at the risk-free rate, and pocket the difference risk-free.


This mispricing cannot persist: as more investors exploit it, buying pressure on the claim and selling pressure on the bond (or direct borrowing) push the claim's price up toward the no-arbitrage value of $1/(1+r_f)$, closing the gap.

## Exercises (match Chapter 3, Suggested Exercises)

1. Compute the bid-ask spread in both dollar and basis-point terms for a stock quoted at a bid of \$49.85 and an ask of \$50.05.
2. A one-period risk-free bond costs \$1,000 and pays \$1,040 in one year. A certain \$1 claim trades at \$0.965. Is there an arbitrage opportunity? If so, describe the trade that exploits it.

In [6]:
# Exercise 1
bid_ex, ask_ex = 49.85, 50.05
spread_ex = ask_ex - bid_ex
mid_ex = (bid_ex + ask_ex) / 2
print(f"Spread: ${spread_ex:.2f} ({spread_ex/mid_ex*10_000:.1f} bps)")

# Exercise 2
rf_ex = (1040 - 1000) / 1000
claim_price_ex = 0.965
no_arb_price_ex = 1 / (1 + rf_ex)
print(f"Risk-free rate: {rf_ex:.2%}")
print(f"No-arbitrage price of $1 claim: ${no_arb_price_ex:.4f}")
print(f"Market price: ${claim_price_ex:.3f} -> "
      f"{'underpriced, buy the claim' if claim_price_ex < no_arb_price_ex else 'overpriced, sell/short the claim'}")

Spread: $0.20 (40.0 bps)
Risk-free rate: 4.00%
No-arbitrage price of $1 claim: $0.9615
Market price: $0.965 -> overpriced, sell/short the claim
